# Intermediate NumPy

**Estimated time:** 90–120 minutes
**Prerequisites:** Basic Python, familiarity with arrays and shape

## Learning Goals

| # | Topic |
|---|-------|
| 1 | Array creation patterns and dtypes |
| 2 | Advanced indexing — fancy, boolean, and `np.ix_` |
| 3 | Broadcasting rules |
| 4 | Vectorization and `np.vectorize` vs ufuncs |
| 5 | Linear algebra — dot products, matrix operations, solving systems |
| 6 | Random number generation (`np.random.default_rng`) |
| 7 | Conditional selection — `np.where` / `np.select` |
| 8 | Performance — views vs copies, memory layout |
| 9 | Transposing, reshaping, `ravel` / `flatten` |
| 10 | Concatenation, stacking, tile, and repeat |
| 11 | Statistical methods and boolean array methods |
| 12 | Sorting and set logic |
| 13 | File I/O — `save`, `load`, `memmap` |
| 14 | Advanced ufunc usage — reduce, accumulate, outer, frompyfunc |
| 15 | Structured and record arrays |
| 16 | ndarray internals — strides, flags, contiguous memory |
| 17 | Numba — just-in-time compilation for NumPy loops |

---

### Quick Reference

```python
a.shape, a.ndim, a.dtype, a.size
np.arange(start, stop, step)
np.linspace(start, stop, n)
a[np.array([0,2,4])]               # fancy index
a[a > 0]                           # boolean index
a @ b                              # matrix multiply
np.linalg.solve(A, b)
rng = np.random.default_rng(seed)
a.T; a.transpose(2,0,1); np.swapaxes(a,0,1)     # transpose
a.ravel(); a.flatten()                           # flatten to 1-D
np.vstack([a,b]); np.hstack([a,b])               # stack
a.sum(axis=0); a.cumsum(); a.argmax()            # stats
np.sort(a); np.argsort(a); np.unique(a)          # sort / set
np.save('f.npy', a); np.load('f.npy')           # file I/O
np.multiply.reduce(a); np.add.accumulate(a)      # ufunc methods
```

In [ ]:
import numpy as np
print('numpy', np.__version__)

---
## Section 1 — Array Creation and Dtypes

NumPy arrays have a fixed dtype. Choosing the right dtype reduces memory and can speed up computation.

| dtype | bytes | range / precision |
|-------|-------|-------------------|
| `int8` | 1 | –128 to 127 |
| `int32` | 4 | –2B to 2B |
| `int64` | 8 | very large integers |
| `float32` | 4 | ~7 decimal digits |
| `float64` | 8 | ~15 decimal digits (default) |
| `bool` | 1 | True/False |

Creating arrays beyond `np.array()`:
- `np.zeros`, `np.ones`, `np.full`, `np.eye`
- `np.arange`, `np.linspace`, `np.logspace`
- `np.empty` — **uninitialized** (fastest, but values are garbage)

In [ ]:
# Different creation strategies
a_int = np.arange(1, 13, dtype=np.int32).reshape(3, 4)
a_float = np.linspace(0, 1, 9).reshape(3, 3)
identity = np.eye(4)
diagonal = np.diag([10, 20, 30, 40])

print('int32 array (3x4):')
print(a_int)
print('\nlinspace float64 (3x3):')
print(a_float.round(3))

In [ ]:
# Memory comparison
n = 1_000_000
arr64 = np.ones(n, dtype=np.float64)
arr32 = np.ones(n, dtype=np.float32)

print(f'float64: {arr64.nbytes / 1e6:.1f} MB')
print(f'float32: {arr32.nbytes / 1e6:.1f} MB')

In [ ]:
# Build a loss development triangle (upper-triangular structure)
# rows = accident years, cols = development ages
triangle = np.array([
    [1_000, 1_500, 1_700, 1_750],
    [  900, 1_350, 1_530,     0],
    [1_200, 1_800,     0,     0],
    [  800,     0,     0,     0],
], dtype=np.float64)

# Replace 0 with NaN for proper masking
triangle[triangle == 0] = np.nan
print(triangle)

In [ ]:
# EXERCISE 1:
# a) Create an array of 20 values log-spaced from 10^0 to 10^4
# b) Create a 5x5 array where entry [i,j] = i * j  (multiplication table)
#    Hint: use np.arange and broadcasting, or np.outer
# YOUR CODE HERE

---
## Section 2 — Advanced Indexing

NumPy has three indexing modes:

| Mode | Syntax | Returns copy or view? |
|------|--------|----------------------|
| Basic (slice) | `a[1:3, 0:2]` | **View** (no copy) |
| Boolean mask | `a[a > 5]` | **Copy** |
| Fancy (integer array) | `a[[0, 2, 4]]` | **Copy** |

`np.ix_` creates an open mesh for selecting sub-matrices with fancy indexing.

In [ ]:
mat = np.arange(25, dtype=float).reshape(5, 5)
print('Full matrix:')
print(mat)

# Basic slice — this is a VIEW
view = mat[1:3, 2:4]
print('\nSlice view [1:3, 2:4]:')
print(view)

# Modifying view changes original!
view[0, 0] = 999
print('\nmat after modifying view:')
print(mat)

In [ ]:
# Reset
mat = np.arange(25, dtype=float).reshape(5, 5)

# Boolean mask — select elements > 10 and set them to 0
mask = mat > 10
print('Mask shape:', mask.shape)
mat_clipped = mat.copy()
mat_clipped[mask] = 0
print(mat_clipped)

In [ ]:
mat = np.arange(25, dtype=float).reshape(5, 5)

# Fancy indexing — pick rows [0,2,4] and cols [1,3]
# np.ix_ creates the right broadcasting shape automatically
rows = np.array([0, 2, 4])
cols = np.array([1, 3])

sub = mat[np.ix_(rows, cols)]  # 3x2 sub-matrix
print('Sub-matrix (rows 0,2,4 x cols 1,3):')
print(sub)

# Compare: without np.ix_ you'd get diagonal pairs, not all combinations
print('\nWithout np.ix_ (element pairs, not sub-matrix):')
print(mat[[0, 2], [1, 3]])  # only 2 elements

In [ ]:
# Practical: extract the latest diagonal from the loss triangle
t = np.array([
    [1_000, 1_500, 1_700, 1_750],
    [  900, 1_350, 1_530,   np.nan],
    [1_200, 1_800,   np.nan, np.nan],
    [  800,   np.nan, np.nan, np.nan],
])

n = t.shape[0]
# Latest diagonal: for row i, take column (n-1-i)
diag_rows = np.arange(n)
diag_cols = n - 1 - diag_rows
latest_diagonal = t[diag_rows, diag_cols]
print('Latest diagonal:', latest_diagonal)

In [ ]:
# EXERCISE 2:
# Given the matrix below, use boolean indexing to:
# a) Find all values that are both > 5 and < 20
# b) Replace all negative values with 0
data = np.array([[ 3, -1, 12, 25],
                 [-5,  8,  0, 18],
                 [14, -3,  7, 22]])
# YOUR CODE HERE

---
## Section 3 — Broadcasting

Broadcasting lets NumPy operate on arrays with **different shapes** by virtually expanding dimensions.

### Rules (applied dimension by dimension, right-to-left)
1. If arrays have different number of dims, prepend 1s to the smaller shape.
2. Dimensions with size 1 are stretched to match the other.
3. If neither is 1 and sizes differ → error.

```
Shape (4, 3) + Shape (3,)   → (4, 3)   ✓  row broadcast
Shape (4, 1) + Shape (1, 3) → (4, 3)   ✓  both broadcast
Shape (4, 3) + Shape (4,)   → ERROR    ✗  last dims 3 ≠ 4
```

In [ ]:
# 1D broadcast: subtract column means from each column
mat = np.random.randint(10, 50, size=(5, 4)).astype(float)
col_means = mat.mean(axis=0)   # shape (4,)

print('Column means:', col_means)
centered = mat - col_means      # (5,4) - (4,) → (5,4)
print('Column means of centered (should be ~0):', centered.mean(axis=0).round(10))

In [ ]:
# Outer product via broadcasting — no np.outer needed
a = np.array([1, 2, 3, 4])      # shape (4,)
b = np.array([10, 20, 30])      # shape (3,)

outer = a[:, np.newaxis] * b[np.newaxis, :]  # (4,1) * (1,3) → (4,3)
print(outer)

In [ ]:
# Actuarial example: apply a different expense load to each row (accident year)
premiums = np.array([1_000_000, 1_200_000, 1_500_000, 1_800_000])  # shape (4,)
expense_loads = np.array([0.28, 0.30, 0.32])                       # shape (3,) — 3 scenarios

# We want net premiums for each (accident_year, scenario) combo
# premiums[:, np.newaxis] is (4,1), expense_loads is (3,) → result (4,3)
net_premium = premiums[:, np.newaxis] * (1 - expense_loads)
print('Net premium (rows=accident years, cols=scenarios):')
print(net_premium.astype(int))

In [ ]:
# EXERCISE 3:
# loss_triangle shape (4 accident years x 4 development ages):
loss = np.array([
    [500, 750, 850, 900],
    [600, 900, 990,   0],
    [700, 980,   0,   0],
    [550,   0,   0,   0],
], dtype=float)
loss[loss == 0] = np.nan

# Using broadcasting:
# a) Compute each cell as a % of that row's maximum (latest observed) value
#    Hint: row max = np.nanmax(loss, axis=1, keepdims=True)  → shape (4,1)
# b) Standardize each column (subtract col mean, divide by col std) — ignore NaN
# YOUR CODE HERE

---
## Section 4 — Vectorization: ufuncs vs np.vectorize

**Universal functions (ufuncs)** are C-level loops: `np.add`, `np.exp`, `np.log`, comparison ops, etc.

`np.vectorize` is a **convenience wrapper** — it still calls Python in a loop and is not truly fast. Use it only when no ufunc alternative exists.

| Approach | Speed | Use when |
|----------|-------|----------|
| ufunc / arithmetic | Fastest | Math on whole arrays |
| `np.where` / `np.select` | Fast | Conditional element-wise |
| `np.vectorize` | Slow (Python loop) | Last resort for custom scalar logic |
| Python `for` loop | Slowest | Never on large arrays |

In [ ]:
x = np.linspace(0.01, 10, 1_000_000)

# ufunc: log-normal distribution PDF — pure NumPy
mu, sigma = 1.5, 0.5
pdf = (1 / (x * sigma * np.sqrt(2 * np.pi))) * np.exp(-((np.log(x) - mu) ** 2) / (2 * sigma**2))
print('PDF computed for 1M points, max:', pdf.max().round(4))

In [ ]:
import math

small_x = np.linspace(0.01, 10, 100_000)

def scalar_pdf(x_val):
    return (1 / (x_val * sigma * math.sqrt(2 * math.pi))) * math.exp(-((math.log(x_val) - mu)**2) / (2 * sigma**2))

vec_pdf = np.vectorize(scalar_pdf)

print('np.vectorize:')
%timeit vec_pdf(small_x)

print('\nNumPy ufuncs:')
%timeit (1/(small_x*sigma*np.sqrt(2*np.pi)))*np.exp(-((np.log(small_x)-mu)**2)/(2*sigma**2))

In [ ]:
# np.select: multi-condition branching without Python loops
loss_ratios = np.array([0.40, 0.62, 0.75, 0.88, 1.10, 0.55])

conditions = [
    loss_ratios < 0.50,
    loss_ratios < 0.70,
    loss_ratios < 0.90,
]
choices = ['Excellent', 'Acceptable', 'Elevated']

rating = np.select(conditions, choices, default='Unacceptable')
print(list(zip(loss_ratios, rating)))

In [ ]:
# EXERCISE 4:
# Compute the present value of a stream of cash flows using vectorized ops
# PV_t = CF_t / (1 + r)^t
cash_flows = np.array([0, 500_000, 750_000, 900_000, 600_000, 400_000])  # t=0..5
r = 0.05
# a) Compute the PV of each cash flow (no loops)
# b) Compute the total NPV
# YOUR CODE HERE

---
## Section 5 — Linear Algebra

Key `np.linalg` functions:

| Function | Purpose |
|----------|---------|
| `a @ b` | Matrix multiply |
| `np.linalg.solve(A, b)` | Solve Ax = b |
| `np.linalg.inv(A)` | Matrix inverse (prefer solve over inv) |
| `np.linalg.det(A)` | Determinant |
| `np.linalg.eig(A)` | Eigenvalues and eigenvectors |
| `np.linalg.lstsq(A, b)` | Least-squares solution |

**Tip:** Avoid `np.linalg.inv` for solving systems — it's slower and less numerically stable than `solve`.

In [ ]:
# Matrix multiply: portfolio return = weights @ returns
weights = np.array([0.4, 0.35, 0.25])          # 3 asset classes
annual_returns = np.array([0.08, 0.05, 0.03])  # bonds, equities, cash

portfolio_return = weights @ annual_returns
print(f'Portfolio return: {portfolio_return:.2%}')

# Covariance matrix (3x3)
cov = np.array([
    [0.04, 0.01, 0.00],
    [0.01, 0.09, 0.01],
    [0.00, 0.01, 0.01],
])

# Portfolio variance = w^T * Cov * w
portfolio_var = weights @ cov @ weights
portfolio_vol = np.sqrt(portfolio_var)
print(f'Portfolio volatility: {portfolio_vol:.2%}')

In [ ]:
# Solve a system of equations — e.g. credibility weighting
# Suppose we have 3 unknowns (loadings) and 3 constraints
A = np.array([
    [1.0, 1.0, 1.0],  # loadings sum to 1
    [2.0, 1.0, 0.5],  # weighted mean constraint
    [1.0, 0.0, 2.0],  # variance constraint
])
b = np.array([1.0, 1.5, 2.0])

x = np.linalg.solve(A, b)
print('Solution x:', x.round(4))
print('Verify Ax = b:', np.allclose(A @ x, b))

In [ ]:
# Least squares: fit a line to noisy data
rng = np.random.default_rng(0)
t = np.arange(10)
y = 2.5 * t + 10 + rng.normal(0, 2, size=10)  # true slope=2.5, intercept=10

# Design matrix [t, 1]
A_ls = np.column_stack([t, np.ones(10)])
coeffs, residuals, rank, sv = np.linalg.lstsq(A_ls, y, rcond=None)
print(f'Fitted slope: {coeffs[0]:.3f}  intercept: {coeffs[1]:.3f}')

In [ ]:
# EXERCISE 5:
# Chain Ladder in NumPy:
# Given the cumulative loss triangle, compute volume-weighted LDFs for each column transition
# LDF(d -> d+1) = sum of column d+1 (observed) / sum of column d (same rows)
tri = np.array([
    [1000, 1500, 1700, 1750],
    [ 900, 1350, 1530,  np.nan],
    [1200, 1800,  np.nan, np.nan],
    [ 800,  np.nan, np.nan, np.nan],
])

# Hint: for transition d -> d+1, only use rows where both columns are observed (not NaN)
# YOUR CODE HERE — compute ldfs as a 1D array of length 3

---
## Section 6 — Random Number Generation

NumPy 1.17+ recommends `np.random.default_rng(seed)` over the legacy `np.random.seed()`. The new Generator API is reproducible, faster, and supports more distributions.

| Method | Distribution |
|--------|--------------|
| `rng.integers(low, high, size)` | Uniform integer |
| `rng.uniform(low, high, size)` | Uniform float |
| `rng.normal(mu, sigma, size)` | Gaussian |
| `rng.lognormal(mu, sigma, size)` | Log-normal |
| `rng.exponential(scale, size)` | Exponential |
| `rng.poisson(lam, size)` | Poisson |
| `rng.gamma(shape, scale, size)` | Gamma |
| `rng.choice(a, size, replace, p)` | Sampling |

In [ ]:
rng = np.random.default_rng(seed=42)

# Simulate 10,000 individual claim severities (log-normal)
claims = rng.lognormal(mean=np.log(25_000), sigma=1.2, size=10_000)
print(f'Mean:   ${claims.mean():,.0f}')
print(f'Median: ${np.median(claims):,.0f}')
print(f'95th %: ${np.percentile(claims, 95):,.0f}')
print(f'99th %: ${np.percentile(claims, 99):,.0f}')

In [ ]:
# Monte Carlo: estimate aggregate loss distribution
# Number of claims ~ Poisson(lam=50)
# Severity ~ LogNormal(mu=log(25000), sigma=1.2)

n_sims = 50_000
rng2 = np.random.default_rng(7)

claim_counts = rng2.poisson(lam=50, size=n_sims)

# Generate all severities at once, then split by count (efficient)
total_claims = claim_counts.sum()
all_severities = rng2.lognormal(np.log(25_000), 1.2, size=total_claims)

# np.split on cumulative counts
splits = np.cumsum(claim_counts)[:-1]
sim_losses = np.array([grp.sum() for grp in np.split(all_severities, splits)])

print('Aggregate loss distribution (50k simulations):')
for p in [50, 75, 90, 95, 99, 99.5]:
    print(f'  {p}th percentile: ${np.percentile(sim_losses, p):>15,.0f}')

In [ ]:
# EXERCISE 6:
# Bootstrap confidence interval for the mean claim severity
# Use the 'claims' array from above (10,000 severities)
# a) Draw 10,000 bootstrap samples of size 200 (with replacement)
# b) Compute the mean of each sample
# c) Report the 95% CI (2.5th and 97.5th percentiles of bootstrap means)
# Hint: rng.choice(claims, size=(10_000, 200), replace=True).mean(axis=1)
# YOUR CODE HERE

---
## Section 7 — np.where and np.select

These replace conditional logic over arrays without Python loops.

```python
np.where(condition, x, y)         # binary: if True → x, else → y
np.select([c1,c2,c3], [v1,v2,v3], default=v_else)  # multi-way
np.clip(a, a_min, a_max)          # cap values
np.nan_to_num(a, nan=0)           # replace NaN
```

In [ ]:
# Apply a reinsurance per-occurrence limit + retention
gross_losses = np.array([50_000, 120_000, 450_000, 1_200_000, 80_000, 2_500_000])
retention = 500_000

# Net loss = min(gross, retention)
net_losses = np.where(gross_losses > retention, retention, gross_losses)
# Reinsurance recovery = gross - net
ri_recovery = gross_losses - net_losses

for g, n, r in zip(gross_losses, net_losses, ri_recovery):
    print(f'Gross: {g:>10,}  Net: {n:>10,}  RI: {r:>10,}')

In [ ]:
# Working with NaN in triangles
tri = np.array([
    [1000., 1500., 1700., 1750.],
    [ 900., 1350., 1530.,   np.nan],
    [1200., 1800.,   np.nan, np.nan],
    [ 800.,   np.nan, np.nan, np.nan],
])

# Compute age-to-age factors only for observed transitions
# Shift: next = tri[:, 1:], prior = tri[:, :-1]
prior = tri[:, :-1]
nxt   = tri[:, 1:]

# Link ratios where both are observed
link_ratios = np.where(~np.isnan(nxt), nxt / prior, np.nan)
print('Link ratios:')
print(link_ratios.round(4))

# Volume-weighted LDFs
ldfs = np.nansum(nxt, axis=0) / np.nansum(prior, axis=0)
print('\nVolume-weighted LDFs:', ldfs.round(4))

In [ ]:
# EXERCISE 7:
# Using ldfs and tri from above:
# a) Compute cumulative development factors (CDFs) to ultimate
#    CDF[i] = product of ldfs[i:] (cumprod from right)
#    Hint: np.cumprod on the reversed array, then reverse back
#    Assume tail factor = 1.0
# b) Project each accident year's ultimate = latest_diagonal * CDF
# YOUR CODE HERE

---
## Section 8 — Views vs Copies and Memory Layout

Understanding views vs copies prevents subtle bugs and reduces memory allocation.

| Operation | View or Copy? |
|-----------|---------------|
| `a[1:3]` (basic slice) | View |
| `a[[0,2]]` (fancy index) | Copy |
| `a[a>0]` (boolean mask) | Copy |
| `a.reshape(...)` | View (usually) |
| `a.T` | View |
| `a.flatten()` | Copy |
| `a.ravel()` | View (when possible) |

**C-order (row-major)** is NumPy's default — rows are contiguous. Transposing gives Fortran order; iterating over columns of a C-order array is slower.

In [ ]:
# Check if an array owns its data
a = np.arange(12).reshape(3, 4)
view = a[0:2, :]      # slice → view
copy = a[[0, 1], :]   # fancy → copy

print('view.base is a:', view.base is a)  # True
print('copy.base is a:', copy.base is a)  # False
print('copy.base is None:', copy.base is None)  # True

# Safer: use .copy() explicitly when you don't want aliasing
safe_copy = a[0:2, :].copy()
print('safe_copy.base is a:', safe_copy.base is a)  # False

In [ ]:
# C-order (row-major) vs F-order (column-major) performance
big_c = np.random.rand(2000, 2000)                     # C-order default
big_f = np.asfortranarray(np.random.rand(2000, 2000))  # F-order

print('Summing along axis=1 (rows):')
%timeit big_c.sum(axis=1)    # contiguous read for C-order
%timeit big_f.sum(axis=1)    # strided read for F-order

print('\nSumming along axis=0 (cols):')
%timeit big_c.sum(axis=0)    # strided for C-order
%timeit big_f.sum(axis=0)    # contiguous for F-order

---
## Section 9 — Transposing, Reshaping, and Flattening

Rearranging the shape or axes of an array without (usually) copying data.

| Operation | Method | Copy? |
|-----------|--------|-------|
| Transpose 2-D | `a.T` | View |
| Transpose n-D | `a.transpose(axes)` | View |
| Swap two axes | `np.swapaxes(a, ax1, ax2)` | View |
| Flatten (always copy) | `a.flatten()` | Copy |
| Flatten (prefer view) | `a.ravel()` | View when possible |
| Reshape with inferred dim | `a.reshape(3, -1)` | View when possible |

**Tip:** Prefer `ravel()` over `flatten()` — it avoids unnecessary allocation when the array is already contiguous.

In [ ]:
# Transpose and axis manipulation
a = np.arange(24).reshape(4, 6)
print('Original shape:', a.shape)
print('a.T shape:     ', a.T.shape)

# .transpose() takes the new axis order explicitly
cube = np.arange(24).reshape(2, 3, 4)  # e.g. (batch, height, width)
print('\ncube shape:', cube.shape)
print('transpose(2,0,1) → (width, batch, height):', cube.transpose(2, 0, 1).shape)

# swapaxes: swap just two axes
print('swapaxes(0,1):', np.swapaxes(cube, 0, 1).shape)

# Verify .T is a view (no copy)
print('\na.T.base is a:', a.T.base is a)  # True

In [ ]:
# ravel vs flatten — same values, different ownership
mat = np.array([[1, 2, 3], [4, 5, 6]])
flat_r = mat.ravel()     # view when C-contiguous
flat_f = mat.flatten()   # always a copy

print('ravel base is mat:  ', flat_r.base is mat)   # True
print('flatten base is mat:', flat_f.base is mat)   # False

# Reshape with -1 as an inferred dimension
arr = np.arange(60)
print('\nOriginal:', arr.shape)
print('reshape(3, -1):', arr.reshape(3, -1).shape)    # (3, 20)
print('reshape(-1, 6):', arr.reshape(-1, 6).shape)    # (10, 6)
print('reshape(2, 5, -1):', arr.reshape(2, 5, -1).shape)  # (2, 5, 6)

# Practical: extract observed values from a triangle (ignore NaN)
tri = np.array([[1., 2., 3.], [4., 5., np.nan], [6., np.nan, np.nan]])
observed = tri.ravel()
observed = observed[~np.isnan(observed)]
print('\nObserved values from triangle:', observed)

In [ ]:
# EXERCISE 9:
# Given a 3-D array representing (policies x years x quarters):
data = np.arange(48).reshape(4, 3, 4)

# a) Transpose to (quarters x years x policies) — axis order (2, 1, 0)
# b) Use swapaxes to swap the policies and years axes (0 and 1)
# c) Flatten data to 1-D with ravel(), then reshape to (12, 4)
# YOUR CODE HERE

---
## Section 10 — Concatenation, Stacking, Tile, and Repeat

Combining and tiling arrays.

| Function | Purpose |
|----------|---------|
| `np.concatenate([a, b], axis=0)` | Join along an existing axis |
| `np.vstack([a, b])` | Stack vertically (axis=0) |
| `np.hstack([a, b])` | Stack horizontally (axis=1) |
| `np.split(a, n, axis)` | Split into n equal pieces |
| `np.r_[a, b]`, `np.c_[a, b]` | Row-wise / column-wise assembly shorthand |
| `np.tile(a, reps)` | Repeat the **whole array** N times |
| `np.repeat(a, n, axis)` | Repeat each **element** N times |
| `arr.take(indices, axis)` | Axis-aware fancy index (like `arr[idx]`) |
| `arr.put(indices, values)` | In-place fancy assignment into flat index |

In [ ]:
# Stacking sub-triangles into a combined triangle
q1 = np.array([[100., 150.], [120., 180.]])   # 2 AY x 2 dev ages
q2 = np.array([[200., 300.], [220., 330.]])

stacked_v = np.vstack([q1, q2])   # same as np.concatenate([q1, q2], axis=0)
stacked_h = np.hstack([q1, q2])   # same as np.concatenate([q1, q2], axis=1)

print('vstack (4x2):\n', stacked_v)
print('\nhstack (2x4):\n', stacked_h)

# r_ and c_ — shorthand for 1-D to 2-D assembly
col_stack = np.c_[np.arange(4), np.arange(4) ** 2]   # (4, 2): [i, i²]
print('\nnp.c_ column stack:\n', col_stack)

# split: reverse of concatenate
parts = np.split(stacked_v, 2, axis=0)
print('\nAfter split, shapes:', [p.shape for p in parts])

In [ ]:
# tile vs repeat
a = np.array([1, 2, 3])
print('np.tile(a, 3)   :', np.tile(a, 3))    # [1,2,3,1,2,3,1,2,3]
print('np.repeat(a, 3) :', np.repeat(a, 3))  # [1,1,1,2,2,2,3,3,3]

# 2-D tile: repeat the whole block
block = np.array([[1, 2], [3, 4]])
print('\nnp.tile(block, (2, 3)):')
print(np.tile(block, (2, 3)))

# take and put — axis-aware fancy indexing
mat = np.arange(12).reshape(3, 4)
print('\ntake rows [0, 2]:\n', mat.take([0, 2], axis=0))

# put: assign to flat index positions
flat = np.zeros(10, dtype=int)
flat.put([1, 3, 5], [99, 88, 77])
print('after put:', flat)

In [ ]:
# EXERCISE 10:
ay2021 = np.array([500., 750., 820., 850.])
ay2022 = np.array([620., 930., 1010., np.nan])
ay2023 = np.array([700., 1050., np.nan, np.nan])

# a) Use vstack to stack the three arrays into a (3, 4) triangle
tri = np.vstack([ay2021, ay2022, ay2023])

# b) Use np.tile to create a (3, 8) array — tile the triangle twice horizontally
# c) Use np.repeat to repeat each element of ay2021 twice → [500,500,750,750,...]
# d) Use take to extract columns 0 and 2 from tri (along axis=1)
# YOUR CODE HERE

---
## Section 11 — Statistical Methods and Boolean Array Methods

These are available as instance methods (`arr.sum()`) **and** as top-level functions (`np.sum(arr)`).

| Method | Description |
|--------|-------------|
| `a.sum(axis)` / `a.mean(axis)` | Sum / mean |
| `a.std(axis)` / `a.var(axis)` | Std deviation / variance |
| `a.min(axis)` / `a.max(axis)` | Min / max |
| `a.argmin(axis)` / `a.argmax(axis)` | Index of min / max |
| `a.cumsum(axis)` | Cumulative sum |
| `a.cumprod(axis)` | Cumulative product |
| `a.any()` | `True` if **any** element is truthy |
| `a.all()` | `True` if **all** elements are truthy |

`axis=0` reduces **across rows** (result has fewer rows); `axis=1` reduces **across columns**.

In [ ]:
# Aggregate stats on a loss triangle
losses = np.array([
    [500., 750., 850., 900.],
    [600., 900., 990., np.nan],
    [700., 980., np.nan, np.nan],
    [550., np.nan, np.nan, np.nan],
])

print('Row sums (ultimate per AY):  ', np.nansum(losses, axis=1))
print('Col means (avg by dev age):  ', np.nanmean(losses, axis=0).round(1))
print('Overall max:                 ', np.nanmax(losses))
print('AY with largest row sum:     ', np.nansum(losses, axis=1).argmax())

# cumsum along a row → cumulative development
print('\nCumulative sum AY0:', losses[0].cumsum())

# cumprod: chain-multiply link ratios to get CDFs
ldfs = np.array([1.50, 1.13, 1.06])
print('CDFs via cumprod:', np.cumprod(ldfs).round(4))

In [ ]:
# any() and all() — concise validation checks
tri = np.array([
    [500., 750., 850., 900.],
    [600., 900., 990., np.nan],
    [700., 980., np.nan, np.nan],
    [550., np.nan, np.nan, np.nan],
])

# Which rows are fully observed (no NaN)?
row_complete = ~np.isnan(tri).any(axis=1)
print('Fully observed rows:', np.where(row_complete)[0])

# Are all latest-diagonal values positive?
n = tri.shape[0]
diagonal = tri[np.arange(n), n - 1 - np.arange(n)]
print('All diagonal values positive:', (diagonal > 0).all())

# any() with a condition — check whether any loss exceeds 900
print('Any loss > 900:', (tri > 900).any())
print('Any loss > 900 per row:', (tri > 900).any(axis=1))

In [ ]:
# EXERCISE 11:
losses = np.array([
    [500., 750., 850., 900.],
    [600., 900., 990., np.nan],
    [700., 980., np.nan, np.nan],
    [550., np.nan, np.nan, np.nan],
])

# a) Find the index of the AY with the largest row sum (nansum + argmax)
# b) Compute the column-wise coefficient of variation: nanstd / nanmean for each dev age
# c) Use all() to confirm that every row has at least one finite value
# d) Compute the running CDF from these period factors using cumprod:
period_factors = np.array([1.48, 1.12, 1.05, 1.02])
# YOUR CODE HERE

---
## Section 12 — Sorting and Set Logic

| Function / Method | Description |
|-------------------|-------------|
| `np.sort(a, axis)` | Returns a sorted **copy** |
| `a.sort(axis)` | Sorts **in-place** |
| `np.argsort(a)` | Indices that would sort `a` |
| `np.lexsort((k1, k2))` | Multi-key sort (last key = primary) |
| `np.partition(a, k)` | Rearranges so k-th element is in final position |
| `np.searchsorted(a, v)` | Binary search in a sorted array |
| `np.unique(a)` | Sorted unique elements |
| `np.in1d(a, b)` | Element-wise membership test |
| `np.union1d(a, b)` | Sorted union |
| `np.intersect1d(a, b)` | Sorted intersection |
| `np.setdiff1d(a, b)` | Elements in `a` not in `b` |

In [ ]:
# Sorting: copy vs in-place
claims = np.array([45_000, 12_000, 280_000, 8_500, 150_000, 62_000])
print('Sorted (copy):', np.sort(claims))

# argsort: get the ordering indices (useful for reordering other arrays)
order = np.argsort(claims)
print('Indices:', order)
print('Claims in order:', claims[order])

# lexsort: multi-key sort — last key in tuple is PRIMARY
claim_ids = np.array([5, 2, 8, 1, 6, 3])
# sort primarily by severity descending, then by id ascending
idx = np.lexsort((claim_ids, -claims))
print('\nlexsort by -severity, then id:')
print(list(zip(claim_ids[idx], claims[idx])))

# partition: find the 3 smallest without full sort (O(n) average)
k = 3
part_idx = np.argpartition(claims, k)[:k]
print('\n3 smallest claims:', np.sort(claims[part_idx]))

# searchsorted: O(log n) lookup in a sorted array
thresholds = np.array([10_000, 50_000, 100_000, 250_000, 500_000])
new_claim = 75_000
bin_idx = np.searchsorted(thresholds, new_claim)
print(f'\nClaim ${new_claim:,} falls in bin index {bin_idx}  (≥{thresholds[bin_idx-1]:,})')

In [ ]:
# Set operations — comparing claim/policy lists across periods
open_2022 = np.array([101, 205, 310, 412, 518, 623])
open_2023 = np.array([205, 412, 715, 820, 101])

# unique: deduplicate a combined list
combined = np.concatenate([open_2022, open_2023])
print('All unique claims:', np.unique(combined))

# in1d: which 2022 claims are still open in 2023?
still_open = open_2022[np.in1d(open_2022, open_2023)]
print('Still open in 2023:', still_open)

# setdiff1d: closed in 2023 (were open in 2022, not in 2023)
closed = np.setdiff1d(open_2022, open_2023)
print('Closed in 2023:', closed)

# setdiff1d: new in 2023
new_claims = np.setdiff1d(open_2023, open_2022)
print('New in 2023:', new_claims)

# union and intersection
print('Ever open (union):', np.union1d(open_2022, open_2023))
print('Open in both years (intersect):', np.intersect1d(open_2022, open_2023))

In [ ]:
# EXERCISE 12:
severities = np.array([23_000, 5_500, 180_000, 42_000, 8_200, 95_000, 312_000, 17_500])
policy_ids = np.array([    'A',   'C',     'A',   'B',   'B',    'C',     'A',    'B'])

# a) Sort severities descending and print the top-3 claim amounts
# b) Use argsort on severities to find the policy_id of the largest claim
# c) How many claims fall in each band? Use searchsorted with:
bands = np.array([0, 25_000, 100_000, 500_000])
# d) Which policy IDs from this book also appear in:
other_policies = np.array(['B', 'D', 'E', 'A'])
#    Use intersect1d on np.unique(policy_ids) and other_policies
# YOUR CODE HERE

---
## Section 13 — File Input and Output with Arrays

| Function | Format | Notes |
|----------|--------|-------|
| `np.save('file.npy', arr)` | Binary `.npy` | Single array, fast |
| `np.load('file.npy')` | Binary `.npy` | Returns ndarray |
| `np.savez('file.npz', a=x, b=y)` | Zipped binary | Multiple named arrays |
| `np.savez_compressed(...)` | Compressed zip | Smaller file, slower write |
| `np.load('file.npz')` | `.npz` | Returns dict-like `NpzFile` |
| `np.savetxt` / `np.loadtxt` | CSV / text | Slow; prefer pandas for CSVs |
| `np.memmap(file, dtype, mode, shape)` | Binary mmap | Out-of-core arrays |

**HDF5:** For very large structured datasets prefer `h5py` or `pandas.HDFStore` over `.npz`.

In [ ]:
import tempfile, os

triangle = np.array([
    [1_000., 1_500., 1_700., 1_750.],
    [  900., 1_350., 1_530.,    np.nan],
    [1_200., 1_800.,    np.nan, np.nan],
    [  800.,    np.nan, np.nan, np.nan],
])

with tempfile.TemporaryDirectory() as tmpdir:
    # Save and reload a single array
    path_npy = os.path.join(tmpdir, 'triangle.npy')
    np.save(path_npy, triangle)
    loaded = np.load(path_npy)
    print('Loaded matches original:', np.array_equal(triangle, loaded, equal_nan=True))
    print('File size (bytes):', os.path.getsize(path_npy))

    # Save multiple arrays in one .npz archive
    ldfs = np.array([1.50, 1.13, 1.06])
    path_npz = os.path.join(tmpdir, 'reserving.npz')
    np.savez(path_npz, triangle=triangle, ldfs=ldfs)

    archive = np.load(path_npz)
    print('\n.npz keys:', list(archive.keys()))
    print('ldfs from archive:', archive['ldfs'])

    # savez_compressed: same API, smaller file
    path_npzc = os.path.join(tmpdir, 'reserving_c.npz')
    np.savez_compressed(path_npzc, triangle=triangle, ldfs=ldfs)
    print(f'\nuncompressed: {os.path.getsize(path_npz):,} bytes')
    print(f'compressed:   {os.path.getsize(path_npzc):,} bytes')

In [ ]:
import tempfile, os

# Memory-mapped file: create a large on-disk array without loading it all into RAM
with tempfile.TemporaryDirectory() as tmpdir:
    mmap_path = os.path.join(tmpdir, 'big_sims.dat')

    # Create a writable memmap — like a file-backed ndarray
    fp = np.memmap(mmap_path, dtype=np.float32, mode='w+', shape=(5_000, 500))
    rng = np.random.default_rng(0)
    fp[:] = rng.random((5_000, 500), dtype=np.float32)
    fp.flush()  # write to disk

    file_mb = os.path.getsize(mmap_path) / 1e6
    print(f'On-disk size: {file_mb:.1f} MB  (5000 x 500 x 4 bytes)')

    # Open read-only — NumPy does not load the full array into RAM at once
    ro = np.memmap(mmap_path, dtype=np.float32, mode='r', shape=(5_000, 500))
    print('Row 0 mean (read from disk):', ro[0].mean().round(4))
    # Slices trigger actual disk reads; unaccessed pages stay on disk

In [ ]:
# EXERCISE 13:
import tempfile, os
rng = np.random.default_rng(99)
sim_losses = rng.lognormal(mean=10.5, sigma=1.0, size=(5_000, 12))  # 5000 sims x 12 months

# a) Save sim_losses to a .npy file in a temp directory and reload it
# b) Compute the monthly 99th percentile from the reloaded array (axis=0) → shape (12,)
# c) Save both sim_losses and the monthly 99th percentiles into a single .npz archive
# d) Load the archive and confirm both keys are present and shapes are correct
# YOUR CODE HERE

---
## Section 14 — Advanced ufunc Usage

Every ufunc (`np.add`, `np.multiply`, `np.maximum`, …) exposes additional reduction methods beyond the basic element-wise call.

| Method | Syntax | Description |
|--------|--------|-------------|
| `reduce` | `np.add.reduce(a)` | Apply op cumulatively, return final scalar |
| `accumulate` | `np.add.accumulate(a)` | Apply op cumulatively, return all intermediates |
| `outer` | `np.multiply.outer(a, b)` | Outer product with any ufunc operation |
| `reduceat` | `np.add.reduceat(a, idx)` | Segmented reduce at given indices |
| `frompyfunc` | `np.frompyfunc(f, nin, nout)` | Wrap a Python scalar fn as a ufunc |

**`np.frompyfunc`** returns an object-dtype result; cast with `.astype(float)` for numeric output.

In [ ]:
# reduce: apply binary op cumulatively → one scalar
ldfs = np.array([1.50, 1.13, 1.06, 1.02])
total_dev = np.multiply.reduce(ldfs)
print('Total development factor (reduce):  ', round(total_dev, 4))

# accumulate: running result → all intermediates
cdfs = np.multiply.accumulate(ldfs)
print('Running CDFs from left (accumulate):', cdfs.round(4))

# add.accumulate is equivalent to cumsum
values = np.array([100., 200., 150., 300.])
print('Running totals:', np.add.accumulate(values))

# outer: every pairwise combination with any ufunc
limits     = np.array([100_000, 250_000, 500_000])
retentions = np.array([ 10_000,  25_000,  50_000])
excess = np.subtract.outer(limits, retentions)
print('\nExcess layer matrix (limits × retentions):')
print(excess)

# reduceat: segment sums — sum within groups defined by index breaks
data = np.array([1., 2., 3., 10., 20., 100., 200.])
breaks = np.array([0, 3, 5])   # groups: [0:3], [3:5], [5:]
print('\nSegment sums:', np.add.reduceat(data, breaks))

In [ ]:
# np.frompyfunc: wrap a scalar Python function as a ufunc
# Useful when no existing ufunc captures your logic
def credibility_blend(ldf, cred):
    # Blend an LDF toward 1.0 using a credibility weight
    return cred * ldf + (1.0 - cred) * 1.0

# 2 inputs, 1 output
blend_ufunc = np.frompyfunc(credibility_blend, 2, 1)

ldf_arr  = np.array([1.50, 1.13, 1.06])
cred_arr = np.array([0.80, 0.90, 0.95])

# Element-wise (like any ufunc) — returns object dtype, cast to float
blended = blend_ufunc(ldf_arr, cred_arr).astype(float)
print('Blended LDFs:', blended.round(4))

# outer: every (ldf, cred) combination
cred_grid = np.array([0.5, 0.75, 1.0])
outer_blend = blend_ufunc.outer(ldf_arr, cred_grid).astype(float)
print('\nOuter blend (ldfs × credibility levels):')
print(outer_blend.round(4))

In [ ]:
# EXERCISE 14:
factors = np.array([1.40, 1.20, 1.08, 1.03, 1.01])

# a) Use np.multiply.reduce to compute the overall CDF (product of all factors)
# b) Use np.multiply.accumulate to get the running CDF at each development stage
# c) Use np.subtract.outer to produce a (5×5) matrix of pairwise factor differences
# d) Write a Python function age_weight(x, n) that returns x / n,
#    wrap it with np.frompyfunc(age_weight, 2, 1), and apply element-wise to:
ages = np.array([1., 2., 3., 4., 5.])
n_periods = np.float64(5.0)
# YOUR CODE HERE

---
## Section 15 — Structured and Record Arrays

Structured arrays store multiple named fields of **different dtypes** in a single ndarray — a lightweight alternative to pandas for fixed-schema tabular data.

```python
dt = np.dtype([('field1', dtype1), ('field2', dtype2), ...])
arr = np.zeros(n, dtype=dt)
arr['field1']   # access entire column
arr[0]          # access first record (structured scalar)
arr[['f1','f2']] # select subset of fields
```

**When to use:** Compact binary storage, interfacing with C structs, or when you need columnar access without the pandas overhead.

In [ ]:
# Define a dtype for a claim record
claim_dtype = np.dtype([
    ('claim_id',    np.int32),
    ('accident_yr', np.int16),
    ('severity',    np.float64),
    ('is_open',     np.bool_),
])

claims = np.zeros(5, dtype=claim_dtype)
claims['claim_id']    = [1001, 1002, 1003, 1004, 1005]
claims['accident_yr'] = [2021, 2021, 2022, 2022, 2023]
claims['severity']    = [45_000, 12_000, 280_000, 8_500, 150_000]
claims['is_open']     = [True, False, True, False, True]

print('All claims:')
print(claims)

# Column access
print('\nSeverities:', claims['severity'])

# Boolean mask on a field
open_claims = claims[claims['is_open']]
print('\nOpen claims (id, severity):')
print(open_claims[['claim_id', 'severity']])

# Sort by severity
sorted_claims = np.sort(claims, order='severity')
print('\nSorted by severity:')
for c in sorted_claims:
    status = 'open' if c['is_open'] else 'closed'
    print(f"  #{c['claim_id']}  AY{c['accident_yr']}  ${c['severity']:>10,.0f}  {status}")

In [ ]:
# EXERCISE 15:
# Create a structured array for policy records with fields:
#   policy_id (int32), premium (float64), loss_ratio (float32), territory (3-char string 'U3')
# a) Populate it with at least 5 records
# b) Retrieve all policies where loss_ratio > 0.70
# c) Sort by premium descending
#    Hint: sort ascending then reverse, or use np.sort(..., order='premium')[::-1]
# YOUR CODE HERE

---
## Section 16 — ndarray Internals: Strides and the Data Buffer

Every ndarray is a view into a 1-D byte buffer. **Strides** describe how many bytes to advance along each axis.

```python
a.strides   # tuple of byte-steps per dimension
a.flags     # C_CONTIGUOUS, F_CONTIGUOUS, OWNDATA, WRITEABLE, …
```

| Concept | Meaning |
|---------|---------|
| `a.strides` | Byte-step tuple for each axis |
| `C_CONTIGUOUS` | Row-major (NumPy default) — rows are stored contiguously |
| `F_CONTIGUOUS` | Column-major (Fortran) — columns are contiguous |
| `np.ascontiguousarray(a)` | Force a C-contiguous copy when needed |

**Tip:** Non-contiguous arrays (e.g., `a.T`) can be slower to iterate over because cache lines aren't fully exploited. Check `flags` before performance-critical work.

In [ ]:
# Inspect strides for different shapes
a = np.arange(12, dtype=np.float64).reshape(3, 4)
print('Shape:', a.shape)
print('Strides (bytes):', a.strides)
print('  → step 32 bytes to next row, 8 bytes to next column (float64 = 8 bytes each)')

# After transpose: strides are swapped, data buffer is the SAME (view)
aT = a.T
print('\nTransposed strides:', aT.strides)         # (8, 32)
print('a.T is C_CONTIGUOUS:', aT.flags['C_CONTIGUOUS'])  # False
print('a.T is F_CONTIGUOUS:', aT.flags['F_CONTIGUOUS'])  # True

# Force C-contiguous copy (needed for some C extensions / Cython code)
aT_c = np.ascontiguousarray(aT)
print('\nForced C-contiguous strides:', aT_c.strides)
print('Now C_CONTIGUOUS:', aT_c.flags['C_CONTIGUOUS'])

# Stride tricks: sliding window view (advanced, read-only)
from numpy.lib.stride_tricks import as_strided
x = np.arange(8, dtype=np.float64)
# 3-element rolling window — shape (6, 3), advancing 1 element (8 bytes) per row
windows = as_strided(x, shape=(6, 3), strides=(8, 8))
print('\n3-element rolling windows:')
print(windows)

In [ ]:
# EXERCISE 16:
a = np.arange(24, dtype=np.float32).reshape(4, 6)

# a) Print a.strides and explain in a comment what each value represents
# b) Check whether a, a.T, and a[:, ::2] are C-contiguous and F-contiguous
# c) Use as_strided from numpy.lib.stride_tricks to create a 2-element
#    sliding window view over the first row of a → shape should be (5, 2)
from numpy.lib.stride_tricks import as_strided
# YOUR CODE HERE

---
## Section 17 — Numba: Just-in-Time Compilation

**Numba** compiles Python/NumPy functions to native machine code at first call. For inner loops that can't be expressed as ufuncs, Numba can match C speed.

```bash
pip install numba   # required; not included with NumPy
```

| Decorator | Use case |
|-----------|----------|
| `@numba.njit` | Compile a scalar / array function to native code |
| `@numba.vectorize(['float64(float64, float64)'])` | Create a true NumPy ufunc |

**When to use Numba:**
- Custom recurrence relations (e.g., iterative triangle projection) that can't be vectorized
- Inner loops over large arrays where `np.vectorize` is too slow
- When Cython or C extensions would otherwise be needed

In [ ]:
try:
    import numba

    @numba.njit
    def project_triangle(triangle, ldfs):
        # Fill the lower-right of a loss triangle using LDFs
        n = triangle.shape[0]
        result = triangle.copy()
        for col in range(1, n):
            ldf = ldfs[col - 1]
            for row in range(n - col, n):
                result[row, col] = result[row, col - 1] * ldf
        return result

    tri = np.array([
        [1000., 1500., 1700., 1750.],
        [ 900., 1350., 1530.,    0.],
        [1200., 1800.,    0.,    0.],
        [ 800.,    0.,    0.,    0.],
    ])
    ldfs = np.array([1.50, 1.13, 1.06])
    projected = project_triangle(tri, ldfs)
    print('Projected triangle (Numba):')
    print(projected.round(0))

    # @numba.vectorize creates a true NumPy ufunc from a scalar function
    @numba.vectorize(['float64(float64, float64)'])
    def credibility_blend(ldf, cred):
        return cred * ldf + (1.0 - cred) * 1.0

    ldf_arr  = np.array([1.50, 1.13, 1.06])
    cred_arr = np.array([0.80, 0.90, 0.95])
    print('\nBlended LDFs (Numba ufunc):', credibility_blend(ldf_arr, cred_arr).round(4))

except ImportError:
    print('numba not installed — run:  pip install numba')
    print('Numba JIT-compiles Python/NumPy loops to native machine code for C-level speed.')

In [ ]:
# EXERCISE 17:
# (Requires numba installed; if not, write the function in plain NumPy)
# a) Write a @numba.njit function cumulative_max(arr) that computes the
#    running maximum of a 1-D float64 array (equivalent to np.maximum.accumulate).
# b) Apply it to:
test_arr = np.array([3., 1., 4., 1., 5., 9., 2., 6.])
# c) Verify your result matches np.maximum.accumulate(test_arr)
# YOUR CODE HERE

---
## Wrap-Up Cheat Sheet

| Topic | Key takeaway |
|-------|--------------|
| dtypes | Match dtype to data — `float32` halves memory vs `float64` |
| Indexing | Slices = views (fast, aliased); fancy/boolean = copies |
| Broadcasting | Align shapes right-to-left; size-1 dims stretch |
| Vectorization | Avoid Python loops — use ufuncs, `np.where`, `np.select` |
| Linear algebra | Prefer `solve` over `inv`; use `@` for matmul |
| RNG | Use `default_rng(seed)` for reproducibility |
| NaN | `np.nansum`, `np.nanmean` skip NaN; `np.isnan` for masking |
| Views | Modifying a view changes the original — use `.copy()` deliberately |
| Transpose | `.T` = view; `transpose(axes)` for n-D; `swapaxes` for two axes |
| Reshape / flatten | `ravel()` prefers view; `flatten()` always copies; use `-1` to infer a dim |
| Stacking | `vstack`/`hstack` = axis-0/1 `concatenate`; `r_`/`c_` are shorthand |
| Tile / repeat | `tile` repeats the whole array; `repeat` repeats each element |
| Stats methods | `cumsum`, `cumprod`, `argmin/max`, `any()`, `all()` — pass `axis=` to aggregate |
| Sorting | `argsort` for indirect sort; `lexsort` for multi-key; `partition` for top-k |
| Set logic | `unique`, `in1d`, `union1d`, `intersect1d`, `setdiff1d` |
| File I/O | `save`/`load` for single arrays; `savez` for multiple; `memmap` for out-of-core |
| ufunc methods | `reduce`, `accumulate`, `outer` work on any ufunc; `frompyfunc` wraps scalar fns |
| Structured arrays | Named-field dtypes for compact tabular storage; access columns by field name |
| Strides | `a.strides` = bytes per axis step; non-contiguous arrays can be slower |
| Numba | `@njit` JIT-compiles loops to C speed; `@vectorize` creates true ufuncs |